## HOW TO upgrade the kit version (internal use only)

This is an interactive notebook to help you update the extscache file in isaac sim to match create.  IsaacSim ships with all creat's plugins.

You need a create daily, and to upgrade to the kit in that, you need that version of kit checked out as well.

#### 1) Checkout create daily branch in `create_folder`

#### 2) Checkout the version of kit in create's `deps/kit-sdk.packman.xml` file in `kit_folder`, e.g. if version is `105.1+master.117022.5b785062.tc.${platform}.${config}` then you can do `git checkout 5b785062`.

#### 3) Set `create_folder` and `kit_folder` as aboce, and set the `isaac_sim_folder` to the repo in which you want to update isaac-sim's kit version.

#### 4) Run the next cell.

In [1]:
kit_folder = "/home/mcarlson/gitlab-master/Graphene"
create_folder = "/home/mcarlson/gitlab-master/ov-create"
isaac_sim_folder = "/home/mcarlson/gitlab-master/mcarlson_omni_isaac_sim"
#get a list of their versions
import os
import toml

def get_exact_version(their_file):
    # file function gets the versions of extensions in a toml's [settings.app.exts] enabled = [ section
    
    their_dict = {}
    for line in open(their_file):
        if line[:8] == "# \tomni.":
            dep = line.partition('-')[0][3:]
            ver = line.partition('-')[2][0:-1]
            their_dict[dep] = ver
    return their_dict

# file function gets the versions of extensions in a toml's [settings.app.exts] enabled = [ section
def get_enabled_version(their_file):

    their_toml = toml.load(their_file)
    their_dict = {}
    for what in their_toml['settings']['app']['exts']['enabled']:
        dep = what.partition('-')
        #if dep[0] in my_toml['dependencies']:
        their_dict[dep[0]] = dep[2]
    return their_dict

# this will add a key value pair to a dict unless it is already there.
def add_or_warn(in_dict, key, val):
    if key in in_dict:
        print (key + " already exists in dictionary as " + in_dict[key] + " so can't set it to " + val)
    else:
        in_dict[key] = val

# gets all the files in a folder.  if ext exists, it only gets files of those types.
def get_files_in_folder(folder, append_dir = True, ext = None):
    files = next(os.walk(folder), (None, None, []))[2] 
    out_files = []
    for file in files:
        if ext and file[-len(ext):] != ext:
            continue
        if append_dir:
            out_files.append(folder + file)
        else:
            out_files.append(file)
            
                
    return out_files

# get extensions in a file with line format:
# "omni.tools.pivot" = {}
def get_extensions(my_file):
    out_list = []
    for line in open(my_file):
        if (line[0] == '"'):
            dep_name = line.partition('"')[2].partition('"')[0]
            out_list.append(dep_name)
    return out_list
            
# get the extensions that are build with kit so we don't need their versions...
def get_local_extensions(depot):
    exts = []
    for x in next(os.walk(depot))[1]:
        exts.append(x)
    return exts


def sort_dict(in_dict, ignores):
    sorted_keys = sorted(in_dict)
    out_dict = {}
    for key in sorted_keys:
        if not key in ignores:
            out_dict[key] = in_dict[key]
    return out_dict


def write_deps(out_file, in_dict, com_out, exclude=None, include_only = None):
    if not exclude == None:
        for key in in_dict:
            if not key in exclude:
                if com_out: out_file.write('#')
                out_file.write("\"" + key + "\" = { version = \"" + in_dict[key] +  "\", exact = true}\n")
    if not include_only == None:
        for key in in_dict:
            if key in include_only:
                if com_out: out_file.write('#')
                out_file.write("\"" + key + "\" = { version = \"" + in_dict[key] +  "\", exact = true}\n")

def write_dictionaries(dicts_with_header, extscache_file, windows_only, com_out=[]):
    with open(extscache_file, "w") as out_file:
        out_file.write("[dependencies]")
        idx = 0
        for key in dicts_with_header:
            out_file.write(key)
            write_deps(out_file, dicts_with_header[key], idx in com_out, exclude=windows_only)
            idx = idx + 1
        
        out_file.write('\n[dependencies.\"filter:platform\".\"windows-x86_64\"]')
        
        idx = 0
        for key in dicts_with_header:
            out_file.write(key)
            write_deps(out_file, dicts_with_header[key], idx in com_out, include_only=windows_only)
            idx = idx + 1


In the next cells we take the generated file from ov-create daily build, and grabs the extensions and versions it uses.

It compares the create exts with the ones from the `*.kit` files in `isaac_sim_folder/source/apps`.  It also ignores the `ignores` list below, as well as extensions that exist inside kit or isaac-sim, because, I think, we don't need those because the versions built with the kit source and isaac-sim source are what we want.

Note the `windows_only` list.  It holds extensions that are windows only.

#### 5) Run the Next cell

In [2]:
# ignore these, don't add them
ignores = [
    "omni.physx" # omni.physx.* is not in the extneions manager
    , "omni.isaac." # Obviously don't need this one.
    , "omni.create.tests" # appearst to be build with create only
    , "omni.pip.compute" # build by isaac sim
]
# known windows only deps, may or may not be needed.
windows_only = [
    "omni.kit.window.modifier.titlebar",
    "omni.kit.converter.cad",
    "omni.kit.converter.lib3mf",
    "omni.rtx.shadercache.d3d12",
    "omni.kit.converter.vtk",
    "omni.kit.simscale",
    #"omni.kit.livestream.webrtc",
    "omni.kit.xr.profile.vr",
    "omni.kit.xr.system.steamvr",
    "omni.kit.converter.stl",
    "omni.usd.fileformat.sbsar", #not sure at all about these sbsar
    "omni.kit.property.sbsar"
]

their_file = create_folder + "/source/apps/omni.create.full.kit"
their_trimmed_file = create_folder + "/source/apps/TRIMMEDomni.create.full.kit"
with open(their_trimmed_file, "w") as out_file:
    found_it = False
    for line in open(their_file):
        if line == "# Exact Version dependencies:\n":
            found_it = True
        if found_it == True:
            out_file.write(line)
        

their_dict = get_enabled_version(their_trimmed_file)
exact_dict = get_exact_version(their_trimmed_file)


my_folder = isaac_sim_folder + "/source/apps/"

in_kit105 = get_local_extensions(kit_folder + "/kit/source/extensions")
in_isaacsim = get_local_extensions(isaac_sim_folder + "/source/extensions")

dot_kit_files = get_files_in_folder(my_folder, ext = 'kit')



There will be extensions that are not in create, but that we need in isaac-sim.  These are found by itteratively running the next cell.

If you want to start over, delete all the `add_or_warn` lines, and you will get a printout of the ones you need to look up.

The printouts make it easy to search http://omniexts.nvidia.com/. For the newest published versions of the extensions.

#### 6) Run this next cell untill you have `add_or_warn` lines filled up.  You can tell you are done because no extensions will print out below.

In [3]:
our_dict = {}

add_or_warn(our_dict, "omni.services.transport.server.tcp_socket", "1.0.2")
add_or_warn(our_dict, "semantics.schema.editor", "0.3.3")
add_or_warn(our_dict, "omni.replicator.core", "1.9.7")
add_or_warn(our_dict, "omni.replicator.composer", "1.2.12")
add_or_warn(our_dict, "omni.kit.livestream.native", "1.0.2")

# print all the extensions that we have in .kit files that do not exist in their_dict
ours_and_theirs = {}
# ran this cell a lot to get the contents of the add_or_warn aditions.
for file_name in dot_kit_files:
    if file_name[-13:] == "extscache.kit":
        continue
    print (f'------------------{file_name[-40:]}')
    exts = get_extensions(file_name)
    for ext in exts:
        do_ignore = False
        for igs in ignores:
            if ext[:len(igs)] == igs:
                do_ignore = True
                break
        if ext in in_kit105 or ext in in_isaacsim or do_ignore: continue
        if ext in their_dict:
            ours_and_theirs[ext] = their_dict[ext]
        if ext not in their_dict and ext not in our_dict and ext not in exact_dict:
            print (f'http://omniexts.nvidia.com/{ext}')
            print (f'add_or_warn(our_dict, "{ext}", "")')

------------------ource/apps/omni.isaac.sim.xr.steamvr.kit
------------------/apps/omni.isaac.sim.headless.webrtc.kit
------------------isaac_sim/source/apps/omni.isaac.sim.kit
------------------/source/apps/omni.isaac.sim.test_ext.kit
------------------isaac_sim/source/apps/omni.app.uidoc.kit
------------------im/source/apps/omni.isaac.sim.splash.kit
------------------_sim/source/apps/omni.isaac.sim.base.kit
------------------im/source/apps/omni.isaac.sim.python.kit
------------------/apps/omni.isaac.sim.headless.native.kit
------------------/source/apps/omni.isaac.sim.selector.kit
------------------_sim/source/apps/exts.deps.generated.kit
------------------isaac.sim.python.gym.headless.render.kit
------------------ps/omni.isaac.sim.headless.websocket.kit
------------------im/source/apps/omni.isaac.sim.fabric.kit
------------------s/omni.isaac.sim.python.gym.headless.kit


Once you have the extension versions sorted out, you run this next cell to actually create the extscache file for isaac sim... it is a descructive creation.  Be warned.
# DESTRUCTIVE OPERATION
#### 7) Run the next cell.   

`com_out` is a list of indices into `dicts_with_header` which you want to have commented out.  This is useful, for example, if create is adding a bunch of extscache deps that are not available on all platforms and builds, but you still want to have an extscache that works for isaacsim.

In [4]:
## Building the contents of source/apps/omni.isaac.sim.extscache.kit
extscache_file = isaac_sim_folder + "/source/apps/omni.isaac.sim.extscache.kit"


their_dict = sort_dict(their_dict, ignores)
exact_dict = sort_dict(exact_dict, ignores)
our_dict = sort_dict(our_dict, ignores)

# Get a create only, and 
needed_for_isaacsim = {}
needed_for_create_only = {}

for ext in their_dict:
    if ext in ours_and_theirs:
        needed_for_isaacsim[ext] = their_dict[ext]
    else:
        needed_for_create_only[ext] = their_dict[ext]
        

dicts_with_header = {
    "\n# Only for Isaac Sim\n":our_dict,
    "\n# Isaac Sim and Create\n":needed_for_isaacsim,
    "\n# Only In Create\n":needed_for_create_only,
    "\n# Exact Version dependencies: In Create\n":exact_dict
}
# index of the dicts we want to comment out
com_out = [2,3]
write_dictionaries(dicts_with_header, extscache_file, windows_only, com_out)


#### 8) Run `git clean -fdx` in the isaac sim repo.

#### 9) Set the kit version in `deps/kit-sdk.packman.xml` to what you want.

#### 10) Build isaac-sim

Step 10 should add comments and a `[settings.app.exts]` section with `enabled = [ SOMETHING ]`.  If SOMETHING is empty, you are done.

Note, If there are extensions that do not exist, you may need to change their versions, or the omniexts servers may be down.

If SOMETHING is not blank the next thing to do is to grab those extensions needed and add them to the file and delete the bottom part so it gets regenerated on next build... to ensure that there are no more "enabled" extensions... if there are, run the next cell again.

#### 11) Run next cell if necessary.

In [5]:
# TODO Make sure all the hand corrections made from the last cell are carried over.  So probably just don't 
# destroy the file and only add to it.

enabled_dict = get_enabled_version(extscache_file)
enabled_dict = sort_dict(enabled_dict, ignores)
dicts_with_header["\n# In Enabled\n"] = enabled_dict


write_dictionaries(dicts_with_header, extscache_file, windows_only, com_out)

#### 12) TODO Some of the exact deps in windows are problematic, you may have to update them when you do with windows build.